In [15]:
!pip install newspaper3k
!pip install sumy
!pip install transformers
!pip install torch
!pip install nltk
!pip install openai
!pip install -U transformers
!pip install -U torch

In [16]:
from newspaper import Article

url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

article = Article(url)
article.download()
article.parse()

text = article.text
print(text[:1000])  # preview

June 13, 2025, 2:33 p.m. ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell hope that it is not your testimony," Turner dug in.

"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.

President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. He has insisted that acquiring Greenland is necessary for national security, citing growing Chinese and Russian influence in the region. The island is also rich in cri

In [17]:
""" 1. Extract and print an extractive summary using the TextRank method."""
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import nltk

nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve the LookupError for punkt_tab

parser = PlaintextParser.from_string(text, Tokenizer("english"))
summarizer = TextRankSummarizer()

print("TextRank Summary:\n")

for sentence in summarizer(parser.document, 5):
    print(sentence)


TextRank Summary:

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
The island is also rich in critical minerals that the U.S. wants to challenge Chinese monopolies in some industries, USA TODAY has reported.
During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force.
In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
""" 2. Extract and print an extractive summary using a frequency-based sentence scoring method."""
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import defaultdict

nltk.download('punkt')
nltk.download('stopwords')

sentences = sent_tokenize(text)

words = word_tokenize(text.lower())
stop_words = set(stopwords.words("english"))

word_freq = defaultdict(int)

for word in words:
    if word.isalnum() and word not in stop_words:
        word_freq[word] += 1

sentence_scores = {}

for sent in sentences:
    for word in word_tokenize(sent.lower()):
        if word in word_freq:
            sentence_scores[sent] = sentence_scores.get(sent, 0) + word_freq[word]

# Top 5 sentences
summary = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:5]

print("Frequency Based Summary:\n")

for s in summary:
    print(s)

Frequency Based Summary:

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.
In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
""" 3. Generate and print an abstractive summary using a pretrained transformer model (e.g., BART or T5). """

from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

inputs = tokenizer(text[:3000], return_tensors="pt", max_length=1024, truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Abstractive Summary:\n")
print(summary)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Abstractive Summary:

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.


In [20]:
""" 4. Print a Lead-3 summary (the first three sentences of the article). """
sentences = sent_tokenize(text)

lead3 = sentences[:3]

print("Lead-3 Summary:\n")

for s in lead3:
    print(s)

Lead-3 Summary:

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


In [21]:
""" 5. Print a manual compression summary, limiting the result to about 20% of the original sentences. """
import math

sentences = sent_tokenize(text)

num_sentences = math.ceil(len(sentences) * 0.20)

compressed_summary = sentences[:num_sentences]

print("Manual Compression Summary (20%):\n")

for s in compressed_summary:
    print(s)

Manual Compression Summary (20%):

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


In [22]:
""" 6. Use an LLM to summary the text. """
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

input_text = "Summarize the following article:\n" + text[:2000]

inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("LLM Summary:\n")
print(summary)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM Summary:

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion.
